# Methodology

How the dataset in this repository was assembled, and the decisions made along the way.

## Scope

- **Time range targeted:** 10 years, 2015–2024.
- **Actually covered:**
  - Penn Clery: 2017–2024 (8 years; 2015–16 not in Penn's current archive — see "Known gaps" below).
  - Penn Pennsylvania UCR: 2017–2024 (8 years).
  - PPD citywide homicides: 2007–2025 YTD (longer for context).
  - UPennAlerts: 2019–2021 only (partial — see "Known gaps").

## Sources

| Series | Source | Format |
|--------|--------|--------|
| Penn Clery (Patrol Zone counts) | Penn Annual Security & Fire Safety Reports (2020, 2022, 2025) | PDF |
| Penn PA UCR (Part 1 offenses + FTE) | Penn Public Safety annual filings, 2017–2024 | PDF |
| PPD citywide homicides | Philadelphia Police Department crime stats / Open Data Philly | CSV / PDF |
| UPennAlerts | Internal Penn alert log, 2019–2021 (partial archive) | Web archive |

Full URLs and retrieval information live in `docs/SOURCES.md` and `data/data_sources.csv`.

## Extraction approach

1. **Manual table extraction from PDFs.** Penn's ASRs publish their crime tables as PDF tables, not machine-readable data. Tables were transcribed by hand from the source PDFs into CSV format.
2. **Cross-checking against multiple ASRs.** Each year of Clery data appears in three consecutive ASRs (the year reported, plus two later annual updates). When figures differ between ASRs, we use the most recent ASR's number — see "Revisions" below.
3. **PA UCR pulled from Penn's annual state filings.** These are the source-of-truth for FTE-normalized counts.
4. **PPD homicides pulled from PPD's published year-end totals**, with 2025 reflecting the most recent YTD count at time of compilation.

## Revisions: which ASR wins

Crime counts often get revised after initial publication as cases reclassify, additional reports come in, or definitions tighten. Each ASR republishes the prior two years' counts alongside the new year. Where a number was revised between editions, we use the most recently published figure as authoritative.

The clearest example is 2019. The 2020 ASR (covering 2017–2019) reports one set of figures for 2019. The 2022 ASR (covering 2019–2021) revises some of those numbers downward. We use the 2022 ASR figures.

## Geography: kept separate, never blended

The four geographies (Penn Patrol Zone, Clery campus, PA UCR area, PPD citywide) cover different physical territories with different reporting rules. We do not:

- Compute a single "Penn area crime rate" by combining sources.
- Calculate growth rates that span boundaries.
- Use citywide context numbers as a denominator for Penn-specific incidents.

Each CSV stays in its own geographic frame. The XLSX output preserves this separation across sheets. Cross-geography comparisons (where we make them in the findings) are explicitly labeled as such.

## Normalization decisions

- **Counts, not rates, where possible.** Rates require denominators, and denominators in this domain are noisier than the numerators. PA UCR is the exception (it's reported per 100k FTE), and there we preserve both the count and the FTE denominator so users can recompute as needed.
- **Categories as published.** We preserve Penn's Clery category names exactly (e.g., "Aggravated Assault," "Motor Vehicle Theft") rather than collapsing into broader buckets.
- **No imputation.** Missing values stay missing. We do not interpolate across the 2015–16 Clery gap; we flag it and link to the federal database where backfill is available.

## Known gaps

1. **2015–16 Clery data.** The 2018 ASR would have covered these years. It is not in Penn's current public archive at time of retrieval. Federal-level data is available via the U.S. Department of Education's Campus Safety database (`ope.ed.gov/campussafety/`). We chose not to backfill from federal data without verifying it matches Penn's later-reported numbers; the gap is flagged in the CSVs with a `note` column.
2. **UPennAlerts pre-2019 and post-2021.** Penn does not maintain a full public archive of UPennAlerts notifications. The 2019–2021 file is a partial extraction from accessible archives. Treat counts as a floor, not a true total.
3. **Hate crime category breakdowns.** Penn reports hate crimes in aggregate but does not consistently disclose the bias category (race, religion, sexual orientation, etc.) in the public ASR tables. Only totals are captured here.
4. **Disciplinary referrals classification.** "Liquor Disciplinary" is reported separately from "Liquor Arrests," and the boundary between them is policy-driven. We capture both where reported.

## Reproducibility

```bash
# 1. Clone repo
git clone <this-repo>
cd upenn-crime-data

# 2. Install dependencies
pip install -r requirements.txt

# 3. Rebuild the compiled XLSX from CSVs
python build_xlsx.py

# 4. Open the exploratory notebook
jupyter notebook notebooks/01_explore.ipynb
```

The CSVs are the source of truth. The XLSX is regenerated. The slide deck (`deliverables/upenn_crime_findings.pptx`) is regenerated from `deliverables/build_deck.js`.

## What this methodology is NOT

- It is not an academic study. There is no peer review, no IRB, no statistical hypothesis testing.
- It is not a journalistic investigation. We didn't interview victims, officers, or administrators.
- It is not a real-time tracker. Numbers reflect official annual reporting cycles, with the lags those imply (Clery year N publishes in October of year N+1).

It is a clean, sourced, reproducible compilation of public data that anyone with a Python install can verify, extend, or argue with.